# Skin Lesion XAI-Trust -- Colab Training/Evaluation Run

Runs the full pipeline from `skin-lesion-xai-trust`: dataset prep, training (hybrid CNN-Transformer + 3 baselines), and evaluation (classification metrics, multi-method XAI faithfulness, MC-Dropout uncertainty, Trust Score).

**Before running:** Runtime -> Change runtime type -> GPU. You will be asked to upload your own `kaggle.json` API token (from kaggle.com -> Settings -> Create New Token) -- it is used only locally in this Colab session, never sent anywhere else.

All results (`runs/*/best.pt`, `runs/eval_report_*.json`) are copied to Google Drive at the end so they survive session disconnects.

## 1. Clone the repository

In [ ]:
!git clone https://github.com/DivyaKathirvelan98/skin-lesion-xai-trust.git
%cd skin-lesion-xai-trust
!pip install -q -r requirements.txt

## 1b. Verify GPU is actually active before proceeding

pip-installing requirements can occasionally pull in a CPU-only torch build over Colab's pre-installed CUDA one. This cell stops the notebook immediately if no GPU is detected, rather than letting you discover it after hours of CPU training.

In [ ]:
import torch
print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device:', torch.cuda.get_device_name(0))
else:
    print()
    print('WARNING: No GPU detected. Training will run on CPU and take many hours.')
    print('Fix: Runtime -> Change runtime type -> Hardware accelerator -> GPU, then')
    print('Runtime -> Restart session, and re-run from the top.')
    raise RuntimeError('No GPU available -- stop and switch runtime type before proceeding.')


## 2. Download HAM10000 (full dataset) directly from Harvard Dataverse

No login or API token needed -- this is the official, fully public dataset release (doi:10.7910/DVN/DBW86T), the same source used to verify the local pilot results. Downloads ~2.78GB: both image archives (10,015 images total), metadata, and the dermatologist-curated ground-truth segmentation masks.

In [ ]:
import os
os.makedirs('data/raw_downloads', exist_ok=True)
os.makedirs('data/raw/images', exist_ok=True)
os.makedirs('data/raw/segmentations', exist_ok=True)

# Official Harvard Dataverse file IDs for doi:10.7910/DVN/DBW86T (verified against the
# dataset's own file listing -- see docs/results.md for how these were confirmed).
files = {
    'HAM10000_metadata.tab': 4338392,
    'HAM10000_segmentations_lesion_tschandl.zip': 3838943,
    'HAM10000_images_part_1.zip': 3172585,
    'HAM10000_images_part_2.zip': 3172584,
}
for fname, file_id in files.items():
    dest = f'data/raw_downloads/{fname}'
    if not os.path.exists(dest):
        print(f'Downloading {fname} ...')
        !curl -L -o "{dest}" "https://dataverse.harvard.edu/api/access/datafile/{file_id}"
    else:
        print(f'{fname} already downloaded, skipping')


In [ ]:
!unzip -q data/raw_downloads/HAM10000_images_part_1.zip -d data/raw/_part1
!unzip -q data/raw_downloads/HAM10000_images_part_2.zip -d data/raw/_part2
!unzip -q data/raw_downloads/HAM10000_segmentations_lesion_tschandl.zip -d data/raw/_seg
!find data/raw/_part1 data/raw/_part2 -name "*.jpg" -exec mv {} data/raw/images/ \;
!find data/raw/_seg -name "*.png" -exec mv {} data/raw/segmentations/ \;
!rm -rf data/raw/_part1 data/raw/_part2 data/raw/_seg
!cp data/raw_downloads/HAM10000_metadata.tab data/raw/HAM10000_metadata_full.tab

import pandas as pd
meta = pd.read_csv('data/raw/HAM10000_metadata_full.tab', sep='	')
meta.to_csv('data/raw/HAM10000_metadata.csv', index=False)
print('images:', len(list(__import__("pathlib").Path("data/raw/images").glob("*.jpg"))))
print('segmentation masks:', len(list(__import__("pathlib").Path("data/raw/segmentations").glob("*.png"))))
print(meta.shape, meta["dx"].value_counts().to_dict())


## 3. Lesion-wise (patient-wise) leakage-safe split

In [ ]:
!python -m src.preprocessing.dataset_split --metadata data/raw/HAM10000_metadata.csv --out data/splits

## 4. Update config to point at the segmentation masks (needed for faithfulness metrics)

In [ ]:
import yaml

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['data']['segmentations_dir'] = 'data/raw/segmentations'
with open('configs/config.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)
print(cfg['data'])

## 5. Train the proposed hybrid model and all three baselines

Reduce `training.epochs` in `configs/config.yaml` first if you need a faster/cheaper run --
the paper should report whatever was actually run, not a target that wasn't executed.

In [ ]:
for model_name in ['hybrid_cnn_transformer', 'resnet50', 'efficientnet_b0', 'vit_base']:
    print(f'=== training {model_name} ===')
    !python -m src.train --config configs/config.yaml --model {model_name}

## 6. Evaluate every model (classification metrics; XAI/uncertainty/Trust Score for the hybrid model)

In [ ]:
for model_name in ['hybrid_cnn_transformer', 'resnet50', 'efficientnet_b0', 'vit_base']:
    print(f'=== evaluating {model_name} ===')
    !python -m src.evaluate --config configs/config.yaml --model {model_name} \
        --checkpoint runs/{model_name}/best.pt --out runs/eval_report_{model_name}.json

## 7. Consolidate results into one table

In [ ]:
import json
import pandas as pd

rows = []
for model_name in ['hybrid_cnn_transformer', 'resnet50', 'efficientnet_b0', 'vit_base']:
    with open(f'runs/eval_report_{model_name}.json') as f:
        report = json.load(f)['report']
    report['model'] = model_name
    rows.append(report)

results_df = pd.DataFrame(rows).set_index('model')
results_df.to_csv('runs/results_summary.csv')
results_df

## 8. Copy results to Google Drive (survives session disconnects)

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
dest = '/content/drive/MyDrive/skin-lesion-xai-trust-results'
shutil.copytree('runs', dest, dirs_exist_ok=True)
print(f'Copied results to {dest}')